### Model 2: 2D CNN on mel spectrograms
using same heartbeat data as model 1

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve,
    recall_score, f1_score, precision_score,
    precision_recall_curve, roc_auc_score, average_precision_score
)
import tensorflow as tf
from tensorflow import keras
from keras import layers, models



In [2]:
import torch
print(torch.version.cuda)

12.8


In [3]:
from google.colab import drive
drive.mount('/content/drive')
DataPath = "drive/MyDrive/JHU/MLMA/final project" # Change to your path
%cd $DataPath

Mounted at /content/drive
/content/drive/MyDrive/JHU/MLMA/final project


In [4]:
!pwd

/content/drive/MyDrive/JHU/MLMA/final project


In [5]:
X_train_full = np.load('X_train.npy')
X_test = np.load('X_test.npy')
y_train_full = np.load('y_train.npy')
y_test = np.load('y_test.npy')
patient_ids_train = np.load('train_patient_ids.npy', allow_pickle=True)
patient_ids_test = np.load('test_patient_ids.npy', allow_pickle=True)
locations_train = np.load('train_locations.npy', allow_pickle=True)
locations_test = np.load('test_locations.npy', allow_pickle=True)

print(f"X_train_full shape: {X_train_full.shape}")  # (46733, 4000)
print(f"X_test shape: {X_test.shape}")              # (12275, 4000)
print(f"y_train_full distribution: {np.bincount(y_train_full)}")
print(f"y_test distribution: {np.bincount(y_test)}")


X_train_full shape: (46733, 4000)
X_test shape: (12275, 4000)
y_train_full distribution: [36770  9963]
y_test distribution: [9890 2385]


### Convert heartbeats to Mel spectrograms

In [6]:

SAMPLE_RATE = 4000
N_MELS = 64
N_FFT = 256
HOP_LENGTH = 64

def heartbeat_to_melspec(signal, sr=SAMPLE_RATE):
    mel_spec = librosa.feature.melspectrogram(
        y=signal.astype(float),
        sr=sr,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        fmin=20,
        fmax=1000
    )
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    return mel_spec_db

X_train_specs = np.array([heartbeat_to_melspec(x) for x in tqdm(X_train_full)])

X_test_specs = np.array([heartbeat_to_melspec(x) for x in tqdm(X_test)])

print(f"X_train_specs shape: {X_train_specs.shape}")  # (46733, 64, 63)
print(f"X_test_specs shape: {X_test_specs.shape}")    # (12275, 64, 63)

# Add channel dimension for CNN
X_train_specs = X_train_specs[..., np.newaxis]
X_test_specs = X_test_specs[..., np.newaxis]

print(f"Final X_train_specs shape: {X_train_specs.shape}")  # (46733, 64, 63, 1)


100%|██████████| 12275/12275 [00:17<00:00, 698.19it/s]

X_train_specs shape: (46733, 64, 63)
X_test_specs shape: (12275, 64, 63)
Final X_train_specs shape: (46733, 64, 63, 1)


In [7]:

N_FOLDS = 5
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS = 100
MIN_SENSITIVITY = 0.70

input_shape = X_train_specs.shape[1:]  # (64, 63, 1)
print(f"Input shape: {input_shape}")

Input shape: (64, 63, 1)


### helper functions

In [8]:

def normalize_data(X_train, X_val):
    """Normalization"""
    mean = X_train.mean()
    std = X_train.std()
    X_train_norm = (X_train - mean) / std
    X_val_norm = (X_val - mean) / std
    return X_train_norm, X_val_norm, mean, std

def aggregate_to_patient_level(preds, labels, patient_ids):
    """Aggregate heartbeat predictions to patient level """
    df = pd.DataFrame({
        'patient_id': patient_ids,
        'pred': preds,
        'label': labels
    })
    return df.groupby('patient_id').agg({
        'pred': 'max',
        'label': 'max'
    }).reset_index()

def compute_metrics(y_true, y_pred_proba, threshold):
    """Compute  metrics"""
    y_pred = (y_pred_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    try:
        auc = roc_auc_score(y_true, y_pred_proba)
    except:
        auc = 0.5

    try:
        auprc = average_precision_score(y_true, y_pred_proba)
    except:
        auprc = 0.0

    try:
        f1 = f1_score(y_true, y_pred)
    except:
        f1 = 0.0

    return {
        'auc': auc, 'auprc': auprc, 'f1': f1,
        'sensitivity': sensitivity, 'specificity': specificity,
        'threshold': threshold,
        'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn
    }

def get_predictions(model, X, batch_size=32):
    """Get predictions from model"""
    preds = model.predict(X, batch_size=batch_size, verbose=0)
    return preds.squeeze()

In [9]:

def build_mel_cnn(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Block 1
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.GlobalAveragePooling2D(),

        # Classifier
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

test_model = build_mel_cnn(input_shape)
test_model.summary()
del test_model


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 63, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 63, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 64, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 31, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 31, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 32, 31, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 15, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 15, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 15, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 15, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 16, 15, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,889 (398.00 KB)

 Trainable params: 101,441 (396.25 KB)

 Non-trainable params: 448 (1.75 KB)

In [10]:

# Get unique patients for CV
train_patient_df = pd.DataFrame({
    'patient_id': patient_ids_train,
    'label': y_train_full
}).drop_duplicates(subset='patient_id')

train_unique_patients = train_patient_df['patient_id'].values
train_patient_labels = train_patient_df['label'].values

print(f"\nCross-validation on {len(train_unique_patients)} patients")
print(f"  With murmur: {train_patient_labels.sum()}")
print(f"  Without murmur: {(train_patient_labels == 0).sum()}")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

cv_val_probs = []
cv_val_labels = []
cv_val_patient_ids = []
fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_unique_patients, train_patient_labels)):
    print(f'\n{"="*50}')
    print(f'FOLD {fold+1}/{N_FOLDS}')
    print(f'{"="*50}')

    # Get patient IDs for this fold
    fold_train_pids = train_unique_patients[train_idx]
    fold_val_pids = train_unique_patients[val_idx]

    # Create masks for heartbeats
    fold_train_mask = np.isin(patient_ids_train, fold_train_pids)
    fold_val_mask = np.isin(patient_ids_train, fold_val_pids)

    # Split data
    X_fold_train = X_train_specs[fold_train_mask]
    y_fold_train = y_train_full[fold_train_mask]
    X_fold_val = X_train_specs[fold_val_mask]
    y_fold_val = y_train_full[fold_val_mask]
    fold_val_patient_ids = patient_ids_train[fold_val_mask]

    print(f'Train: {len(X_fold_train)} heartbeats from {len(fold_train_pids)} patients')
    print(f'Val:   {len(X_fold_val)} heartbeats from {len(fold_val_pids)} patients')

    # Normalize per fold
    X_fold_train_norm, X_fold_val_norm, fold_mean, fold_std = normalize_data(
        X_fold_train, X_fold_val
    )

    # Compute class weights
    n_neg = (y_fold_train == 0).sum()
    n_pos = (y_fold_train == 1).sum()
    class_weight = {0: 1.0, 1: n_neg / n_pos}
    print(f'Class weight for Present: {class_weight[1]:.2f}')

    # Build fresh model
    model = build_mel_cnn(input_shape)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=15,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=0
        )
    ]

    # Train
    history = model.fit(
        X_fold_train_norm, y_fold_train,
        validation_data=(X_fold_val_norm, y_fold_val),
        epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=0
    )

    print(f'Stopped at epoch {len(history.history["loss"])}')

    # Get validation predictions
    val_probs = get_predictions(model, X_fold_val_norm)

    # Aggregate to patient level
    val_patient_df = aggregate_to_patient_level(
        val_probs, y_fold_val, fold_val_patient_ids
    )

    cv_val_probs.extend(val_patient_df['pred'].values)
    cv_val_labels.extend(val_patient_df['label'].values)
    cv_val_patient_ids.extend(val_patient_df['patient_id'].values)

    # Compute fold metrics
    fold_auroc = roc_auc_score(val_patient_df['label'], val_patient_df['pred'])
    fold_auprc = average_precision_score(val_patient_df['label'], val_patient_df['pred'])

    fold_metrics.append({
        'fold': fold + 1,
        'auroc': fold_auroc,
        'auprc': fold_auprc,
        'n_patients': len(val_patient_df),
        'n_positive': val_patient_df['label'].sum()
    })

    print(f'\nFold {fold+1} Patient-Level Results:')
    print(f'  AUROC: {fold_auroc:.4f}')
    print(f'  AUPRC: {fold_auprc:.4f}')

    # Clean up
    del model
    keras.backend.clear_session()

# Convert to arrays
cv_val_probs = np.array(cv_val_probs)
cv_val_labels = np.array(cv_val_labels)

# Summary
print('\n' + '='*60)
print('CROSS-VALIDATION SUMMARY')
print('='*60)

fold_metrics_df = pd.DataFrame(fold_metrics)
print(f"\nPer-fold results:")
print(fold_metrics_df.to_string(index=False))

mean_auroc = fold_metrics_df['auroc'].mean()
std_auroc = fold_metrics_df['auroc'].std()
mean_auprc = fold_metrics_df['auprc'].mean()
std_auprc = fold_metrics_df['auprc'].std()

print(f"\nMean AUROC: {mean_auroc:.4f} ± {std_auroc:.4f}")
print(f"Mean AUPRC: {mean_auprc:.4f} ± {std_auprc:.4f}")



Cross-validation on 696 patients
  With murmur: 141
  Without murmur: 555

FOLD 1/5
Train: 37550 heartbeats from 556 patients
Val:   9183 heartbeats from 140 patients
Class weight for Present: 3.80
Epoch 32: early stopping
Restoring model weights from the end of the best epoch: 17.
Stopped at epoch 32

Fold 1 Patient-Level Results:
  AUROC: 0.6928
  AUPRC: 0.5337

FOLD 2/5
Train: 37651 heartbeats from 557 patients
Val:   9082 heartbeats from 139 patients
Class weight for Present: 3.57
Epoch 20: early stopping
Restoring model weights from the end of the best epoch: 5.
Stopped at epoch 20

Fold 2 Patient-Level Results:
  AUROC: 0.7455
  AUPRC: 0.5178

FOLD 3/5
Train: 37457 heartbeats from 557 patients
Val:   9276 heartbeats from 139 patients
Class weight for Present: 3.65
Epoch 28: early stopping
Restoring model weights from the end of the best epoch: 13.
Stopped at epoch 28

Fold 3 Patient-Level Results:
  AUROC: 0.8514
  AUPRC: 0.7369

FOLD 4/5
Train: 37043 heartbeats from 557 patient

### threshold selection - based on f1 scores
#### and then train final model that's selected

In [11]:

print("\n" + "="*60)
print("THRESHOLD SELECTION")
print("="*60)

# F1-optimized threshold
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(cv_val_labels, cv_val_probs)
f1_scores = 2 * (precision_curve * recall_curve) / (precision_curve + recall_curve + 1e-8)
best_f1_idx = np.argmax(f1_scores[:-1])

best_threshold = pr_thresholds[best_f1_idx]
print(f"\nF1-Optimized Threshold: {best_threshold:.4f}")
print(f"  Precision: {precision_curve[best_f1_idx]:.4f}")
print(f"  Recall: {recall_curve[best_f1_idx]:.4f}")
print(f"  F1: {f1_scores[best_f1_idx]:.4f}")


print("\n" + "="*60)
print("TRAINING FINAL MODEL")
print("="*60)

# Normalize full training set
X_train_mean = X_train_specs.mean()
X_train_std = X_train_specs.std()
X_train_norm = (X_train_specs - X_train_mean) / X_train_std
X_test_norm = (X_test_specs - X_train_mean) / X_train_std

# Class weights
n_neg = (y_train_full == 0).sum()
n_pos = (y_train_full == 1).sum()
class_weight = {0: 1.0, 1: n_neg / n_pos}

# Build final model
final_model = build_mel_cnn(input_shape)
final_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Small validation split for early stopping
from sklearn.model_selection import train_test_split
val_split_patients, _ = train_test_split(
    train_unique_patients,
    test_size=0.9,
    stratify=train_patient_labels,
    random_state=99
)

final_val_mask = np.isin(patient_ids_train, val_split_patients)
final_train_mask = ~final_val_mask

X_final_train = X_train_norm[final_train_mask]
y_final_train = y_train_full[final_train_mask]
X_final_val = X_train_norm[final_val_mask]
y_final_val = y_train_full[final_val_mask]

print(f"Final training: {len(X_final_train)} heartbeats")
print(f"Final validation: {len(X_final_val)} heartbeats")

import os
os.makedirs('m2', exist_ok=True)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'm2/model2_final.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

history = final_model.fit(
    X_final_train, y_final_train,
    validation_data=(X_final_val, y_final_val),
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

print(f"\nTraining complete! Stopped at epoch {len(history.history['loss'])}")



THRESHOLD SELECTION

F1-Optimized Threshold: 0.8425
  Precision: 0.5877
  Recall: 0.4752
  F1: 0.5255

TRAINING FINAL MODEL
Final training: 42160 heartbeats
Final validation: 4573 heartbeats
Epoch 1/100
1318/1318 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - accuracy: 0.6159 - loss: 1.0448 - val_accuracy: 0.8161 - val_loss: 0.4945 - learning_rate: 1.0000e-04
Epoch 2/100
1318/1318 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.6738 - loss: 0.9797 - val_accuracy: 0.6766 - val_loss: 0.6214 - learning_rate: 1.0000e-04
Epoch 3/100
1318/1318 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.6968 - loss: 0.9519 - val_accuracy: 0.7853 - val_loss: 0.5041 - learning_rate: 1.0000e-04
Epoch 4/100
1318/1318 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.7106 - loss: 0.9293 - val_accuracy: 0.8268 - val_loss: 0.4414 - learning_rate: 1.0000e-04
Epoch 5/100
1318/1318 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.7235 - loss: 0.9094 - val_accuracy: 0.8347 - val_loss: 0.4540 - learning_rate: 1.0000e-04
Epoch 6/100

#### final evaluation on test set

In [12]:


# Get heartbeat-level predictions
test_probs = get_predictions(final_model, X_test_norm)
test_labels = y_test
test_pids_out = patient_ids_test

print(f"\nTest set: {len(X_test)} heartbeats from {len(np.unique(patient_ids_test))} patients")

# Recording/heartbeat-level metrics
print(f"\n--- Heartbeat-Level Metrics ---")
hb_metrics = compute_metrics(test_labels, test_probs, best_threshold)
print(f"AUROC: {hb_metrics['auc']:.4f}")
print(f"Sensitivity: {hb_metrics['sensitivity']:.4f}")
print(f"Specificity: {hb_metrics['specificity']:.4f}")

# Patient-level metrics
print(f"\n--- Patient-Level Metrics (PRIMARY) ---")
test_patient_df = aggregate_to_patient_level(test_probs, test_labels, test_pids_out)

patient_probs = test_patient_df['pred'].values
patient_labels = test_patient_df['label'].values

patient_metrics = compute_metrics(patient_labels, patient_probs, best_threshold)

print(f"Test patients: {len(test_patient_df)}")
print(f"  With murmur: {patient_labels.sum()}")
print(f"  Without murmur: {(patient_labels == 0).sum()}")
print(f"\nAt threshold = {best_threshold:.4f}:")
print(f"  AUROC:       {patient_metrics['auc']:.4f}")
print(f"  AUPRC:       {patient_metrics['auprc']:.4f}")
print(f"  Sensitivity: {patient_metrics['sensitivity']:.4f}")
print(f"  Specificity: {patient_metrics['specificity']:.4f}")
print(f"  F1 Score:    {patient_metrics['f1']:.4f}")
print(f"\nConfusion Matrix:")
print(f"  TP={patient_metrics['tp']}, FP={patient_metrics['fp']}")
print(f"  FN={patient_metrics['fn']}, TN={patient_metrics['tn']}")



Test set: 12275 heartbeats from 175 patients

--- Heartbeat-Level Metrics ---
AUROC: 0.7582
Sensitivity: 0.1686
Specificity: 0.9933

--- Patient-Level Metrics (PRIMARY) ---
Test patients: 175
  With murmur: 36
  Without murmur: 139

At threshold = 0.8425:
  AUROC:       0.7068
  AUPRC:       0.4983
  Sensitivity: 0.6389
  Specificity: 0.7410
  F1 Score:    0.4842

Confusion Matrix:
  TP=23, FP=36
  FN=13, TN=103


In [13]:
#### save results for visualization later

### zone specific analysis

In [14]:

zone_results = []

for location in ['AV', 'PV', 'TV', 'MV']:
    loc_mask = locations_test == location

    if loc_mask.sum() == 0:
        print(f'\n{location}: No samples')
        continue

    loc_probs = test_probs[loc_mask]
    loc_labels = test_labels[loc_mask]
    loc_patient_ids = test_pids_out[loc_mask]

    # Aggregate to patient level
    loc_patient_df = aggregate_to_patient_level(loc_probs, loc_labels, loc_patient_ids)

    # Compute metrics
    loc_metrics = compute_metrics(loc_patient_df['label'], loc_patient_df['pred'], best_threshold)

    print(f'\n{location}:')
    print(f'  Patients: {len(loc_patient_df)} ({int(loc_patient_df["label"].sum())} with murmur)')
    print(f'  AUC:         {loc_metrics["auc"]:.4f}')
    print(f'  AUPRC:       {loc_metrics["auprc"]:.4f}')
    print(f'  Sensitivity: {loc_metrics["sensitivity"]:.4f}')
    print(f'  Specificity: {loc_metrics["specificity"]:.4f}')
    print(f'  F1:          {loc_metrics["f1"]:.4f}')

    zone_results.append({
        'zone': location,
        'n_patients': len(loc_patient_df),
        'n_positive': int(loc_patient_df['label'].sum()),
        'auc': loc_metrics['auc'],
        'auprc': loc_metrics['auprc'],
        'sensitivity': loc_metrics['sensitivity'],
        'specificity': loc_metrics['specificity'],
        'f1': loc_metrics['f1']
    })

# Save zone results
zone_df = pd.DataFrame(zone_results)
zone_df.to_csv('m2/model2_zone_metrics.csv', index=False)
print("\n✓ Saved: m2/model2_zone_metrics.csv")



AV:
  Patients: 157 (29 with murmur)
  AUC:         0.6824
  AUPRC:       0.4320
  Sensitivity: 0.4138
  Specificity: 0.8516
  F1:          0.4000

PV:
  Patients: 146 (29 with murmur)
  AUC:         0.7686
  AUPRC:       0.5565
  Sensitivity: 0.4483
  Specificity: 0.9060
  F1:          0.4906

TV:
  Patients: 138 (26 with murmur)
  AUC:         0.7332
  AUPRC:       0.5378
  Sensitivity: 0.3846
  Specificity: 0.9375
  F1:          0.4651

MV:
  Patients: 159 (33 with murmur)
  AUC:         0.6883
  AUPRC:       0.4255
  Sensitivity: 0.3939
  Specificity: 0.8968
  F1:          0.4407

✓ Saved: m2/model2_zone_metrics.csv


In [15]:
import os
os.makedirs('results', exist_ok=True)

# Patient-level
np.save('results/model2_patient_probs.npy', test_patient_df['pred'].values)
np.save('results/model2_patient_labels.npy', test_patient_df['label'].values)
np.save('results/model2_patient_ids.npy', test_patient_df['patient_id'].values)
np.save('results/model2_threshold.npy', np.array([best_threshold]))

# Heartbeat-level (for comparison notebook zone analysis)
np.save('results/model2_hb_probs.npy', test_probs)
np.save('results/model2_hb_labels.npy', test_labels)
np.save('results/model2_hb_patient_ids.npy', test_pids_out)
np.save('results/model2_hb_locations.npy', locations_test)

print("✓ Saved: results/model2_patient_*.npy")
print("✓ Saved: results/model2_hb_*.npy")

# Save model
final_model.save('m2/model2_final.keras')
print("✓ Saved: m2/model2_final.keras")

# Save preprocessing params
preprocessing_params = {
    'X_mean': float(X_train_mean),
    'X_std': float(X_train_std),
    'sample_rate': SAMPLE_RATE,
    'n_mels': N_MELS,
    'hop_length': HOP_LENGTH,
    'n_fft': N_FFT,
    'optimal_threshold': float(best_threshold)
}
np.save('m2/model2_preprocessing_params.npy', preprocessing_params)
print("✓ Saved: m2/model2_preprocessing_params.npy")

# Save CV fold metrics
fold_metrics_df.to_csv('m2/model2_cv_fold_metrics.csv', index=False)
print("✓ Saved: m2/model2_cv_fold_metrics.csv")


✓ Saved: results/model2_patient_*.npy
✓ Saved: results/model2_hb_*.npy
✓ Saved: m2/model2_final.keras
✓ Saved: m2/model2_preprocessing_params.npy
✓ Saved: m2/model2_cv_fold_metrics.csv


In [16]:

print("\n" + "="*60)
print("MODEL 2 COMPLETE - FINAL SUMMARY")
print("="*60)
print(f"Model: 2D CNN on Mel Spectrograms ")
print(f"Input shape: {input_shape}")
print(f"\nCross-Validation:")
print(f"  AUROC: {mean_auroc:.4f} ± {std_auroc:.4f}")
print(f"  AUPRC: {mean_auprc:.4f} ± {std_auprc:.4f}")
print(f"\nThreshold: {best_threshold:.4f} (F1-optimized)")
print(f"\nTest Performance (Patient-Level):")
print(f"  AUROC:       {patient_metrics['auc']:.4f}")
print(f"  AUPRC:       {patient_metrics['auprc']:.4f}")
print(f"  Sensitivity: {patient_metrics['sensitivity']:.4f} ({patient_metrics['tp']}/{patient_metrics['tp']+patient_metrics['fn']})")
print(f"  Specificity: {patient_metrics['specificity']:.4f}")
print(f"  F1 Score:    {patient_metrics['f1']:.4f}")


MODEL 2 COMPLETE - FINAL SUMMARY
Model: 2D CNN on Mel Spectrograms 
Input shape: (64, 63, 1)

Cross-Validation:
  AUROC: 0.7707 ± 0.0670
  AUPRC: 0.5976 ± 0.1293

Threshold: 0.8425 (F1-optimized)

Test Performance (Patient-Level):
  AUROC:       0.7068
  AUPRC:       0.4983
  Sensitivity: 0.6389 (23/36)
  Specificity: 0.7410
  F1 Score:    0.4842
